# Intermediate Machine Learning 
* Datasets from Housing Prices Competition for Kaggle Learn Users on Kaggle

## Lesson 1: Introduction (Random Forest Tuning)

### 1. The Goal
Real-world machine learning involves tweaking models to find the absolute best predictive performance. In this warm-up, we are going to load the Iowa Housing dataset, define five different Random Forest architectures (using different **hyperparameters**), and write an automated loop to test which one performs the best.

### 2. Loading and Splitting the Data
First, we load our data using Pandas. We separate our target variable (`SalePrice`) from our features. 

To ensure we can accurately test our models, we use `train_test_split` to hide 20% of our training data. This hidden 20% becomes our **Validation Set**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the data (Update these paths if you put the CSVs in a different folder)
X_full = pd.read_csv('../data/train.csv', index_col='Id')
X_test_full = pd.read_csv('../data/test.csv', index_col='Id')

# 2. Separate the Target (y) from the Features
y = X_full.SalePrice

# Target - what we are trying to predict
# Features - These are the inputs used to guess the target

# For this warm-up, we are only using 7 specific numeric features
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
X = X_full[features].copy()
X_test = X_test_full[features].copy()

# 3. Break off the validation set from the training data (80% Train / 20% Validate)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# Preview the training features
X_train.head()

,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
Id,,,,,,,
619,11694,2007,1828,0,2,3,9
871,6600,1962,894,0,1,2,5
93,13360,1921,964,0,1,2,5
818,13265,2002,1689,0,2,3,7
303,13704,2001,1541,0,2,3,6


### 3. Defining the Competitors (Hyperparameter Tuning)
A Random Forest has several settings (hyperparameters) we can tweak to prevent underfitting or overfitting:
* **`n_estimators`:** The total number of trees in the forest.
* **`criterion`:** The math formula the tree uses to evaluate its splits (e.g., absolute error vs. squared error).
* **`max_depth`:** The maximum number of splits a tree is allowed to make.
* **`min_samples_split`:** The minimum number of houses required in a node before it is allowed to split again.

We will define 5 different models to race against each other.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Define 5 different Random Forest architectures
model_1 = RandomForestRegressor(n_estimators=50, random_state=0)
model_2 = RandomForestRegressor(n_estimators=100, random_state=0)
model_3 = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)
model_4 = RandomForestRegressor(n_estimators=200, min_samples_split=20, random_state=0)
model_5 = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=0)

# Group them into a list so we can loop through them
models = [model_1, model_2, model_3, model_4, model_5]

### 4. Evaluating the Models
Instead of writing the training and testing code five separate times, we define a custom Python function `score_model()`. We can pass any model into this function, and it will automatically fit the data, make predictions on the validation set, and return the Mean Absolute Error (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error

# 1. Define the evaluation function
def score_model(model, X_t=X_train, X_v=X_valid, y_t=y_train, y_v=y_valid):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return mean_absolute_error(y_v, preds)

# 2. Loop through the 5 models and print their scores
print("Evaluating Model Architectures:\n" + "-"*35)
for i in range(0, len(models)):
    mae = score_model(models[i])
    print(f"Model {i+1} MAE: ${mae:,.0f}")


# The best model is the one with the lowest MAE

Evaluating Model Architectures:
-----------------------------------
Model 1 MAE: $24,015
Model 2 MAE: $23,741
Model 3 MAE: $23,529
Model 4 MAE: $23,997
Model 5 MAE: $23,707


### 5. Training the Final Model for Production
After running the evaluation loop, we observe which model achieved the lowest MAE. For this dataset, **Model 3** (100 trees using `absolute_error` criterion) performs the best.

Now that we know the optimal architecture, we define our final model. Because we are ready to predict on the *actual* test data (the unknown houses), we no longer need to hide 20% of our data for validation. We train this final model on **100% of the available training data (`X` and `y`)** to make it as smart as possible.

In [ ]:
# 1. Define the final model using the winning architecture (Model 3)
my_model = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)

# 2. Fit the model to ALL of the training data
my_model.fit(X, y)

# 3. Generate predictions for the completely unseen test.csv data
preds_test = my_model.predict(X_test)

# 4. Save the predictions to a CSV file (This is the format Kaggle requires for submission)
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': preds_test})
output.to_csv('submission.csv', index=False)

print("Final predictions successfully saved to submission.csv!")

Final predictions successfully saved to submission.csv!


## Intermediate ML - Lesson 2: Missing Values

### 1. Why Do Missing Values Happen?
Real-world datasets are rarely pristine. Missing data occurs for many reasons:
* A 2-bedroom house won't have a value for the size of a 3rd bedroom.
* A survey respondent might decline to share their income.

Most Machine Learning algorithms (including Scikit-Learn's `RandomForestRegressor`) will throw a severe error if you feed them raw data containing missing values (`NaN` or `None`). We need a systematic strategy to clean these before modeling.

---

### 2. Three Strategies for Missing Data

#### Strategy 1: Drop Columns with Missing Values
* **How it works:** Find any column that contains at least one `NaN` and delete the entire column.
* **Pros:** Very fast and simple.
* **Cons:** Loses a massive amount of valuable information. If a column of 10,000 houses is only missing 1 value, dropping the column throws away 9,999 good pieces of data!

#### Strategy 2: Imputation
* **How it works:** Fill in the missing cells with a reasonable guess—usually the average (mean) or median value of that column.
* **Pros:** Retains all rows and keeps the dataset size intact. Almost always yields better predictions than dropping columns.
* **Cons:** The filled-in numbers aren't 100% accurate, which can introduce slight noise.

#### Strategy 3: An Extension to Imputation
* **How it works:** Fill in the missing values (imputation), **plus** add a new boolean column (e.g., `col_was_missing`) indicating which rows were originally blank.
* **Pros:** Gives the model extra context. If missing a specific detail is a signal in itself (e.g., missing garage info means "no garage"), the model can learn from that flag.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the data
X_full = pd.read_csv('train.csv', index_col='Id')
X_test_full = pd.read_csv('test.csv', index_col='Id')

# 2. Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# 3. To keep it simple for now, use only numerical predictors
X = X_full.select_dtypes(exclude=['object'])
X_test = X_test_full.select_dtypes(exclude=['object'])

# 4. Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# Preview training data shape
print(f"Training shape: {X_train.shape}")

### 3. Evaluating Performance Across the 3 Approaches

To objectively test which strategy works best, we define a reusable benchmark function `score_dataset()`. This function trains a standard Random Forest and returns the Mean Absolute Error (MAE).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Helper function to evaluate MAE of a dataset
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

#### Approach 1: Drop Columns with Missing Values

In [ ]:
# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

print("MAE from Approach 1 (Drop columns with missing values):")
print(f"${score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid):,.2f}")

#### Approach 2: Imputation using `SimpleImputer`

In [ ]:
from sklearn.impute import SimpleImputer

# Imputation
my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

# Imputation removes column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

print("MAE from Approach 2 (Imputation):")
print(f"${score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid):,.2f}")

#### Approach 3: An Extension to Imputation (Tracking Imputed Entries)

In [ ]:
# Make copies to avoid changing original data
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

# Add new columns indicating which values were missing
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

# Imputation
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(my_imputer.transform(X_valid_plus))

# Put back column names
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

print("MAE from Approach 3 (Extension to Imputation):")
print(f"${score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid):,.2f}")

### 4. Conclusion & Key Takeaways
1. **Imputation (Approach 2) beat Dropping Columns (Approach 1)** because dropping entire columns removed too much valuable feature signal from the housing data.
2. **When to use extension flags:** Approach 3 provides a slight boost when the *fact* that a value is missing conveys significant extra meaning (e.g., missing `GarageYrBlt` means there is no garage).

## Intermediate ML - Lesson 2: Missing Values

### 1. Overview & Setup
In real-world data, missing values are extremely common. Standard Scikit-Learn models throw errors if fed data containing `NaN` or `None` values.

This exercise explores three approaches to dealing with missing numerical data:
1. **Drop columns** with missing values.
2. **Imputation** (filling missing cells with the column mean).
3. **An Extension to Imputation** (imputing + adding a boolean `_was_missing` flag column).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Read the data
X_full = pd.read_csv('train.csv', index_col='Id')
X_test_full = pd.read_csv('test.csv', index_col='Id')

# 2. Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# 3. Keep only numerical predictors for simplicity
X = X_full.select_dtypes(exclude=['object'])
X_test = X_test_full.select_dtypes(exclude=['object'])

# 4. Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# Preview first few rows
X_train.head()

### 2. Step 1: Preliminary Investigation
Before choosing a handling strategy, we inspect how much data is actually missing.

In [ ]:
# Total rows in training data
num_rows = X_train.shape[0]  # 1168

# Number of columns with missing values
missing_val_count_by_column = (X_train.isnull().sum())
cols_with_missing = missing_val_count_by_column[missing_val_count_by_column > 0]
num_cols_with_missing = len(cols_with_missing)  # 3 columns

# Total missing entries across all columns
tot_missing = cols_with_missing.sum()  # 276 missing values

print(f"Total Rows: {num_rows}")
print(f"Columns with Missing Data: {num_cols_with_missing}")
print(f"Total Missing Entries: {tot_missing}")

### 3. Benchmark Scoring Function
We define `score_dataset()` to evaluate how each missing value strategy impacts Random Forest performance using **Mean Absolute Error (MAE)**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

### 4. Step 2: Drop Columns with Missing Values
We locate every column containing a missing value and drop it completely from both `X_train` and `X_valid`.

In [ ]:
# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]

# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

print("MAE (Drop columns with missing values):")
print(f"${score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid):,.2f}")

### 5. Step 3: Imputation
We use Scikit-Learn's `SimpleImputer` to replace missing cells with the average (mean) value of that column.

In [ ]:
from sklearn.impute import SimpleImputer

# Imputation
my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

# Imputation removes column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

print("MAE (Imputation):")
print(f"${score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid):,.2f}")

### 6. Step 4: Final Model & Test Predictions
Now we preprocess both our training/validation data and the blind `X_test` data using imputation before generating final competition predictions.

In [ ]:
# 1. Preprocess training and validation data using Imputation
final_imputer = SimpleImputer(strategy='median')
final_X_train = pd.DataFrame(final_imputer.fit_transform(X_train), columns=X_train.columns)
final_X_valid = pd.DataFrame(final_imputer.transform(X_valid), columns=X_valid.columns)

# 2. Fit the final model
model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(final_X_train, y_train)

# 3. Preprocess test data and predict
final_X_test = pd.DataFrame(final_imputer.transform(X_test), columns=X_test.columns)
preds_test = model.predict(final_X_test)

# 4. Save test predictions to submission file
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': preds_test})
output.to_csv('submission.csv', index=False)
print("Saved final predictions to submission.csv!")